# Day 12 — SQL vs Pandas Comparison
**Dataset:** Employee_data_v2.csv (474 employees)  
**Columns:** id, gender, bdate, educ, jobcat, salary, salbegin, jobtime, prevexp, attrition

---

## Setup — Load Data

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('Employee_data_v2.csv')

# Clean prevexp: replace 'missing' with NaN and convert to numeric
df['prevexp'] = pd.to_numeric(df['prevexp'], errors='coerce')

print(f'Shape: {df.shape}')
df.head()

---
## PART 1 — Basic SELECT / WHERE Queries

---

### Q1 — All Manager Employees

**SQL:**
```sql
SELECT * FROM employees WHERE jobcat = 'Manager';
```

In [ ]:
# Pandas equivalent
q1 = df[df['jobcat'] == 'Manager']
print(f'Manager count: {len(q1)}')
q1.head()

### Q2 — Employees with Salary > 50000

**SQL:**
```sql
SELECT id, gender, jobcat, salary FROM employees WHERE salary > 50000;
```

In [ ]:
q2 = df[df['salary'] > 50000][['id', 'gender', 'jobcat', 'salary']]
print(f'High earners (>50k): {len(q2)}')
q2.head()

### Q3 — Female Employees Who Have Not Left (attrition = No)

**SQL:**
```sql
SELECT id, gender, jobcat, salary FROM employees
WHERE gender = 'Female' AND attrition = 'No';
```

In [ ]:
q3 = df[(df['gender'] == 'Female') & (df['attrition'] == 'No')][['id', 'gender', 'jobcat', 'salary']]
print(f'Retained female employees: {len(q3)}')
q3.head()

### Q4 — Education >= 16, Sorted by Salary Descending

**SQL:**
```sql
SELECT id, educ, jobcat, salary FROM employees
WHERE educ >= 16 ORDER BY salary DESC;
```

In [ ]:
q4 = df[df['educ'] >= 16][['id', 'educ', 'jobcat', 'salary']].sort_values('salary', ascending=False)
print(f'High education (educ>=16): {len(q4)}')
q4.head()

### Q5 — Starting Salary Below 15000

**SQL:**
```sql
SELECT id, gender, jobcat, salbegin, salary FROM employees
WHERE salbegin < 15000 ORDER BY salbegin ASC;
```

In [ ]:
q5 = df[df['salbegin'] < 15000][['id', 'gender', 'jobcat', 'salbegin', 'salary']].sort_values('salbegin')
print(f'Low starting salary (<15k): {len(q5)}')
q5.head()

---
## PART 2 — JOIN Queries (Simulated with Merge)

---

### Q6 — Self-Join: Employee Pairs in Same JobCat Where emp1 Earns More

**SQL:**
```sql
SELECT a.id AS emp1_id, b.id AS emp2_id, a.jobcat, a.salary AS salary1, b.salary AS salary2
FROM employees a
JOIN employees b ON a.jobcat = b.jobcat AND a.salary > b.salary AND a.id < b.id
LIMIT 20;
```

In [ ]:
# Self merge on jobcat
self_join = df[['id', 'jobcat', 'salary']].merge(
    df[['id', 'jobcat', 'salary']],
    on='jobcat',
    suffixes=('_a', '_b')
)
q6 = self_join[
    (self_join['salary_a'] > self_join['salary_b']) &
    (self_join['id_a'] < self_join['id_b'])
][['id_a', 'id_b', 'jobcat', 'salary_a', 'salary_b']].head(20)
q6

### Q7 — Join with Derived Table: Avg Salary Per JobCat

**SQL:**
```sql
SELECT e.id, e.jobcat, e.salary, avg_table.avg_salary
FROM employees e
JOIN (SELECT jobcat, AVG(salary) AS avg_salary FROM employees GROUP BY jobcat) AS avg_table
ON e.jobcat = avg_table.jobcat;
```

In [ ]:
avg_table = df.groupby('jobcat')['salary'].mean().reset_index()
avg_table.columns = ['jobcat', 'avg_salary']

q7 = df[['id', 'jobcat', 'salary']].merge(avg_table, on='jobcat')
q7['avg_salary'] = q7['avg_salary'].round(2)
q7.head()

### Q8 — Join to Get Top Earner Per Gender

**SQL:**
```sql
SELECT e.id, e.gender, e.salary, top_earners.max_salary
FROM employees e
JOIN (SELECT gender, MAX(salary) AS max_salary FROM employees GROUP BY gender) AS top_earners
ON e.gender = top_earners.gender AND e.salary = top_earners.max_salary;
```

In [ ]:
max_by_gender = df.groupby('gender')['salary'].max().reset_index()
max_by_gender.columns = ['gender', 'max_salary']

q8 = df[['id', 'gender', 'salary']].merge(max_by_gender, on='gender')
q8 = q8[q8['salary'] == q8['max_salary']]
q8

### Q9 — Flag Employees as Above/Below Average in Their Department

**SQL:**
```sql
SELECT e.id, e.jobcat, e.salary, stats.avg_salary,
       CASE WHEN e.salary >= stats.avg_salary THEN 'Above Average' ELSE 'Below Average' END AS salary_status
FROM employees e
JOIN (SELECT jobcat, AVG(salary) AS avg_salary FROM employees GROUP BY jobcat) AS stats
ON e.jobcat = stats.jobcat;
```

In [ ]:
dept_avg = df.groupby('jobcat')['salary'].mean().reset_index()
dept_avg.columns = ['jobcat', 'avg_salary']

q9 = df[['id', 'jobcat', 'salary']].merge(dept_avg, on='jobcat')
q9['salary_status'] = q9.apply(
    lambda row: 'Above Average' if row['salary'] >= row['avg_salary'] else 'Below Average',
    axis=1
)
q9.sort_values(['jobcat', 'salary'], ascending=[True, False]).head(10)

### Q10 — Self-Join: Same Education AND Same JobCat

**SQL:**
```sql
SELECT a.id AS emp1_id, b.id AS emp2_id, a.educ, a.jobcat
FROM employees a
JOIN employees b ON a.educ = b.educ AND a.jobcat = b.jobcat AND a.id < b.id
LIMIT 20;
```

In [ ]:
educ_join = df[['id', 'educ', 'jobcat']].merge(
    df[['id', 'educ', 'jobcat']],
    on=['educ', 'jobcat'],
    suffixes=('_a', '_b')
)
q10 = educ_join[educ_join['id_a'] < educ_join['id_b']][['id_a', 'id_b', 'educ', 'jobcat']].head(20)
q10

---
## PART 3 — Aggregation Queries

---

### Q11 — Count of Employees by Job Category

**SQL:**
```sql
SELECT jobcat, COUNT(*) AS total_employees FROM employees GROUP BY jobcat ORDER BY total_employees DESC;
```

In [ ]:
q11 = df.groupby('jobcat').size().reset_index(name='total_employees').sort_values('total_employees', ascending=False)
q11

### Q12 — Avg Current and Starting Salary by Gender

**SQL:**
```sql
SELECT gender, ROUND(AVG(salary),2) AS avg_current_salary, ROUND(AVG(salbegin),2) AS avg_starting_salary
FROM employees GROUP BY gender;
```

In [ ]:
q12 = df.groupby('gender').agg(
    avg_current_salary=('salary', 'mean'),
    avg_starting_salary=('salbegin', 'mean')
).round(2).reset_index()
q12

### Q13 — Min, Max, Avg Salary by Job Category

**SQL:**
```sql
SELECT jobcat, MIN(salary) AS min_salary, MAX(salary) AS max_salary, ROUND(AVG(salary),2) AS avg_salary
FROM employees GROUP BY jobcat ORDER BY avg_salary DESC;
```

In [ ]:
q13 = df.groupby('jobcat')['salary'].agg(['min', 'max', 'mean']).round(2).reset_index()
q13.columns = ['jobcat', 'min_salary', 'max_salary', 'avg_salary']
q13 = q13.sort_values('avg_salary', ascending=False)
q13

### Q14 — Attrition Count and Percentage by Job Category

**SQL:**
```sql
SELECT jobcat, COUNT(*) AS total,
       SUM(CASE WHEN attrition='Yes' THEN 1 ELSE 0 END) AS left_company,
       ROUND(SUM(CASE WHEN attrition='Yes' THEN 1 ELSE 0 END)*100.0/COUNT(*),2) AS attrition_pct
FROM employees GROUP BY jobcat;
```

In [ ]:
q14 = df.groupby('jobcat').apply(
    lambda x: pd.Series({
        'total': len(x),
        'left_company': (x['attrition'] == 'Yes').sum(),
        'attrition_pct': round((x['attrition'] == 'Yes').sum() / len(x) * 100, 2)
    })
).reset_index()
q14

### Q15 — Avg Education Level by JobCat and Gender

**SQL:**
```sql
SELECT jobcat, gender, ROUND(AVG(educ),2) AS avg_education
FROM employees GROUP BY jobcat, gender ORDER BY jobcat, gender;
```

In [ ]:
q15 = df.groupby(['jobcat', 'gender'])['educ'].mean().round(2).reset_index()
q15.columns = ['jobcat', 'gender', 'avg_education']
q15.sort_values(['jobcat', 'gender'])

---
## PART 4 — Advanced SQL Problems with Pandas Equivalents

---

### A1 — Running Total of Salary (Window Function)

**SQL:**
```sql
SELECT id, jobcat, salary,
       SUM(salary) OVER (
           PARTITION BY jobcat ORDER BY id
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
       ) AS running_total_salary
FROM employees ORDER BY jobcat, id;
```

In [ ]:
a1 = df[['id', 'jobcat', 'salary']].sort_values(['jobcat', 'id'])
a1['running_total_salary'] = a1.groupby('jobcat')['salary'].cumsum()
a1.head(10)

### A2 — Top 3 Earners Per Category using RANK()

**SQL:**
```sql
SELECT * FROM (
    SELECT id, gender, jobcat, salary,
           RANK() OVER (PARTITION BY jobcat ORDER BY salary DESC) AS salary_rank
    FROM employees
) AS ranked
WHERE salary_rank <= 3
ORDER BY jobcat, salary_rank;
```

In [ ]:
a2 = df[['id', 'gender', 'jobcat', 'salary']].copy()
a2['salary_rank'] = a2.groupby('jobcat')['salary'].rank(method='min', ascending=False).astype(int)
a2 = a2[a2['salary_rank'] <= 3].sort_values(['jobcat', 'salary_rank'])
a2

### A3 — Month-over-Month Growth using LAG()

**Concept:** LAG() fetches the value from the previous row within a window. Here we use it to compare consecutive salary entries per job category (ordered by id as hire proxy).

**SQL:**
```sql
SELECT id, jobcat, salary,
       LAG(salary) OVER (PARTITION BY jobcat ORDER BY id) AS prev_salary,
       salary - LAG(salary) OVER (PARTITION BY jobcat ORDER BY id) AS salary_diff,
       ROUND(
           (salary - LAG(salary) OVER (...)) * 100.0
           / NULLIF(LAG(salary) OVER (...), 0), 2
       ) AS pct_change
FROM employees ORDER BY jobcat, id;
```

In [ ]:
a3 = df[['id', 'jobcat', 'salary']].sort_values(['jobcat', 'id']).copy()

# shift(1) = LAG(1) — get previous row within group
a3['prev_salary'] = a3.groupby('jobcat')['salary'].shift(1)
a3['salary_diff'] = a3['salary'] - a3['prev_salary']
a3['pct_change'] = ((a3['salary_diff'] / a3['prev_salary']) * 100).round(2)

a3.dropna(subset=['prev_salary']).head(10)

### A4 — CTE for Multi-Step Calculation

**Concept:** CTEs (Common Table Expressions) define intermediate result sets. Here: Step 1 = dept averages, Step 2 = salary growth per employee, Step 3 = join and label.

**SQL:**
```sql
WITH dept_avg AS (
    SELECT jobcat, AVG(salary) AS avg_salary FROM employees GROUP BY jobcat
),
salary_growth AS (
    SELECT id, gender, jobcat, salary, salbegin,
           ROUND((salary - salbegin)*1.0/salbegin*100, 2) AS growth_pct
    FROM employees
)
SELECT sg.*, da.avg_salary,
       CASE WHEN sg.salary > da.avg_salary THEN 'Above Avg' ELSE 'Below Avg' END AS salary_status
FROM salary_growth sg JOIN dept_avg da ON sg.jobcat = da.jobcat
ORDER BY sg.growth_pct DESC;
```

In [ ]:
# Step 1: CTE dept_avg
dept_avg = df.groupby('jobcat')['salary'].mean().reset_index()
dept_avg.columns = ['jobcat', 'avg_salary']

# Step 2: CTE salary_growth
salary_growth = df[['id', 'gender', 'jobcat', 'salary', 'salbegin']].copy()
salary_growth['growth_pct'] = ((salary_growth['salary'] - salary_growth['salbegin']) / salary_growth['salbegin'] * 100).round(2)

# Step 3: Merge + label
a4 = salary_growth.merge(dept_avg, on='jobcat')
a4['salary_status'] = a4.apply(
    lambda row: 'Above Avg' if row['salary'] > row['avg_salary'] else 'Below Avg', axis=1
)
a4.sort_values('growth_pct', ascending=False).head(10)

### A5 — Correlated Subquery for Employee-Department Comparison

**Concept:** A correlated subquery references the outer query's current row. For each employee, it computes the avg salary of their own job category and checks if the employee earns above it.

**SQL:**
```sql
SELECT id, gender, jobcat, salary
FROM employees e
WHERE salary > (
    SELECT AVG(salary) FROM employees WHERE jobcat = e.jobcat
)
ORDER BY jobcat, salary DESC;
```

In [ ]:
# In Pandas, we replicate the correlated subquery by merging dept avg and filtering
dept_avg_a5 = df.groupby('jobcat')['salary'].mean().reset_index()
dept_avg_a5.columns = ['jobcat', 'dept_avg_salary']

a5 = df[['id', 'gender', 'jobcat', 'salary']].merge(dept_avg_a5, on='jobcat')
a5 = a5[a5['salary'] > a5['dept_avg_salary']].sort_values(['jobcat', 'salary'], ascending=[True, False])
a5 = a5.drop(columns='dept_avg_salary')  # Drop helper column to match SQL output

print(f'Employees earning above their dept average: {len(a5)}')
a5.head(10)